In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.chains import RetrievalQA
from langchain_classic.chains.question_answering import load_qa_chain



In [26]:
import os

In [27]:
os.environ["GOOGLE_API_KEY"] = os.getenv("api_key")

In [28]:
#Load dos modelos

embeddings_models = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview", api_key=os.getenv("api_key"))
llm = ChatGoogleGenerativeAI(model="gemini-3.1-pro-preview")

In [29]:
#carregar pdf

PDF_LINK = "1758887557527-attachment.pdf"
loader = PyPDFLoader(PDF_LINK, extract_images=False)
pages = loader.load_and_split()

#separar chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=4000, 
    chunk_overlap=20,
    length_function=len,
    add_start_index=True
)

chunks = text_splitter.split_documents(pages)



In [30]:
#salvar no VECTOR DB

db = Chroma.from_documents(chunks, embedding=embeddings_models, persist_directory="text_index")



In [ ]:
#carregar db

vectordb = Chroma(persist_directory="text_index", embedding_function=embeddings_models)

#load retriever

retriever = vectordb.as_retriever(search_kwargs={"k": 3})
chain = load_qa_chain(llm, chain_type="stuff")

In [32]:
def ask_question(question):
    context = retriever._get_relevant_documents(question, run_manager=None)
    answer = (chain({"input_documents": context, "query": question}, return_only_outputs=True))["output_text"]
    return answer



In [ ]:
#user_question = input("Digite sua pergunta: ")
#answer = ask_question(user_question)